In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

In [ ]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

In [ ]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

In [ ]:
from dotenv import load_dotenv
import openai
import os
import numpy as np

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

In [ ]:
from tqdm import tqdm
import os, json
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

from tree_of_thought_v2 import TreeOfThoughtExplorer
from tree_of_thought_judge import PathSelectionJudge

from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case


from profiles.schema import PersonSet
from profiles.profile_dict import PERSON_SEEDS


case_name_set = ["stereotype"]      # "manipulation"
max_branching_factor = 3
max_depth = 2
n_shots = 1
max_tokens_dict = {"generation": 500}


role_playing_mode = "none"
selected_profiles = [None]          #  ["profile1","profile2"] for role_playing_mode != "none"
person_set = PersonSet(seeds=PERSON_SEEDS, metadata={})



base_dir = f"results/{model_filename}/tot_explorer"
os.makedirs(base_dir, exist_ok=True)

try:
    for profile_n in selected_profiles:
        for case_name in case_name_set:
            print(f"\n=== Running ToT Explorer + Judge for {case_name} ===")

            if case_name.lower() == "manipulation":
                case = manipulation_case
                task_definition = manipulation_definition_short
                data = sample_mentalmanip
            elif case_name.lower() == "stereotype":
                case = stereotypes_case
                task_definition = stereotype_definition_short_binary
                data = sample_mgsd
            else:
                raise ValueError(f"Unknown case name: {case_name}")

            if role_playing_mode == "none" or profile_n is None:
                out_dir = f"{base_dir}/baseline"
            else:
                out_dir = f"{base_dir}/role_playing/{profile_n}_{role_playing_mode}"
            os.makedirs(out_dir, exist_ok=True)

            output_file = f"{out_dir}/results_{case_name.lower()}_tot_explorer.csv"
            reasoning_file = f"{out_dir}/reasoning_{case_name.lower()}_tot_explorer.json"

            explorer = TreeOfThoughtExplorer(
                case=case,
                client=client,
                model=model,
                max_branching_factor=max_branching_factor,
                max_depth=max_depth,
                task_definition=task_definition,
                max_tokens_dict=max_tokens_dict,
                n_shots=n_shots,
                examples_df=None,
                reasoning_budget=None
            )

            judge = PathSelectionJudge(
                client=client,
                model=model,
                temperature=0.0,
                max_tokens=256,
                person_key=profile_n if role_playing_mode != "none" else None,
                role_playing=role_playing_mode,
                person_set=person_set
            )

            rows = []
            detailed = []

            for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()

                try:
                    paths = explorer.solve(text)
                    selection = judge.choose_best_from_explorer(case=case, explorer_paths=paths, max_paths=12)

                    selected_path_id = selection.get("path_id")
                    selected_label = selection.get("label")

                    if not selected_label:
                        if paths and paths[0] and paths[0][-1].verdict:
                            selected_label = paths[0][-1].verdict
                        else:
                            selected_label = list(case.valid_labels)[-1]

                    mapped_label = case.label_map.get(
                        str(selected_label).strip(), list(case.label_map.values())[-1]
                    )

                    rows.append({
                        "sample_id": idx,
                        "text": text,
                        "true_label": true_label,
                        "pred_label": mapped_label,
                        "raw_pred_label": selected_label,
                        "selected_path_id": selected_path_id,
                        "role_playing": role_playing_mode,
                        "person_key": profile_n if role_playing_mode != "none" else None,
                    })


                    all_paths_serialized = []
                    for p in paths:
                        all_paths_serialized.append({
                            "path_id": "->".join(t.id for t in p),
                            "steps": [
                                {
                                    "id": t.id,
                                    "content": t.content,
                                    "label": t.verdict
                                } for t in p
                            ]
                        })

                    detailed.append({
                        "sample_id": idx,
                        "judge_raw": selection.get("raw"),
                        "selected_path_id": selected_path_id,
                        "selected_label": selected_label,
                        "paths": all_paths_serialized,
                    })

                except Exception as e:
                    print(f"Error processing sample {idx}: {e}")
                    continue

            if not rows:
                print(f"No successful selections for {case_name}")
                continue

            df_out = pd.DataFrame(rows)
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            df_out.to_csv(output_file, index=False)
            print(f"=== Saved {len(df_out)} rows to {output_file} ===")

            os.makedirs(os.path.dirname(reasoning_file), exist_ok=True)
            with open(reasoning_file, "w", encoding="utf-8") as f:
                json.dump(detailed, f, indent=2, ensure_ascii=False)
            print(f"=== Saved detailed reasoning to {reasoning_file} ===")

            try:
                if case_name.lower() == "manipulation":
                    y_true = df_out["true_label"].astype(int)
                    y_pred = df_out["pred_label"].astype(int)
                elif case_name.lower() == "stereotype":
                    y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()

                print(f"\n=== Classification Report for {case_name} (ToT Explorer + Judge) ===")
                print(classification_report(y_true, y_pred, zero_division=0))
                print(f"\n=== Confusion Matrix for {case_name} ===")
                labels = sorted(set(y_true) | set(y_pred))
                print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))

                accuracy = (y_true == y_pred).mean()
                print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")

                print(f"\n=== Label Distribution ===")
                print("True labels:")
                print(pd.Series(y_true).value_counts())
                print("Predicted labels:")
                print(pd.Series(y_pred).value_counts())

            except Exception as e:
                print(f"Error in evaluation for {case_name}: {e}")

            print(f"\n=== Sample Chosen Paths (raw/parsed) ===")
            for i, item in enumerate(detailed[:3]):
                print(f"\nSample {item['sample_id']}:")
                tr = df_out.loc[df_out["sample_id"] == item["sample_id"], "true_label"].values
                print(f"True Label: {tr[0] if len(tr) else 'N/A'}")
                print(f"Selected Label: {item['selected_label']}")
                print(f"Selected Path ID: {item['selected_path_id']}")
                chosen = next((p for p in item["paths"] if p["path_id"] == item["selected_path_id"]), None)
                if chosen:
                    print("Chosen Path Steps:")
                    for step in chosen["steps"]:
                        print(f"  {step['id']}: {step['content'][:180]} (Label: {step.get('label')})")
                else:
                    print("Chosen path not found in serialized paths.")
                print("Judge Raw (truncated):")
                jr = item.get("judge_raw") or ""
                print((jr[:300] + "...") if len(jr) > 300 else jr)
                print("-" * 50)

except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== ToT Explorer + Judge Evaluation Complete ===")
